In [2]:
import pandas as pd
import numpy as np
import re
import json
import nltk
from wordcloud import WordCloud, STOPWORDS
import matplotlib.pyplot as plt
from nltk.stem.porter import PorterStemmer
from textblob import TextBlob
from nltk.corpus import stopwords
from pytesseract import TesseractError
import PIL
import pdb

#nltk.download('stopwords')
stop = stopwords.words('english')
PIL.Image.MAX_IMAGE_PIXELS = 1003445512

In [3]:
def words_clean_cloud_df(df, column_name, stem = False, outputdf = False):

    comment_words = ' '
    stopwords = set(stop) 
    stopwords.update(["New", "York", "Times", "copyright", "would"])
    porter = PorterStemmer()

    # iterate through the csv file 
    for val in df[column_name].str.lower(): 

        # typecaste each val to string 
        #val = str(val) 

        # split the value 
        tokens = val.split() 

        # Converts each token into lowercase 
        #for i in range(len(tokens)): 
            #tokens[i] = tokens[i].lower() 

        if stem is True:
            stemmed = [porter.stem(word) for word in tokens]
            for words in stemmed: 
                comment_words = comment_words + words + ' '
        else:
            for words in tokens: 
                comment_words = comment_words + words + ' '

                
                
                
    wordcloud = WordCloud(width = 1600, height = 900, background_color ='white', 
                    stopwords = stopwords, max_font_size=100, min_font_size=4)
    for line in comment_words:  # Here you can also use the Cursor
        counts_line = wordcloud.process_text(line)
        counts_all.update(counts_line)

    wordcloud.generate_from_frequencies(counts_all)
    wordcloud.to_file(df+'.png')
    
    
    plt.figure(figsize = (16, 8), facecolor = None) 
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis("off") 
    plt.tight_layout(pad = 0) 

    plt.show() 
    
    if outputdf is True and stem is True:
        data = pd.DataFrame(stemmed) 
        for i in stop :
            data = data.replace(to_replace=r'\b%s\b'%i, value=np.NaN, regex=True).dropna()
        data["count"] = 1
        data = data.rename(columns = {0:"words"})
        data_group = data.groupby("words").count()
        data_group = data_group.sort_values(by = "count", ascending = False)
        return data_group
    
    elif outputdf is True and stem is False:
        data = pd.DataFrame(tokens) 
        for i in stop :
            data = data.replace(to_replace=r'\b%s\b'%i, value=np.NaN, regex=True).dropna()
        data["count"] = 1
        data = data.rename(columns = {0:"words"})
        data_group = data.groupby("words").count()
        data_group = data_group.sort_values(by = "count", ascending = False)
        return data_group
    

In [4]:
def remove(text):
    remove_chars = '[0-9’!"#$%&\'()*+,-./:;<=>?@，。?★、…【】《》？“”‘’！[\\]^_`{|}~]+'
    return re.sub(remove_chars, '', text)

In [5]:
filename = "nyt_terrorism_pdftexts_proquest2.csv"
titles = ["index", "pubdate", "text"]
#write_csv_title(filename, titles)

In [6]:
data = pd.read_csv("total_nyt_terrorism_news1850-2019.csv", encoding = 'utf-8')
data["date"] = pd.to_datetime(data['pubdate'], errors = "coerce")

# no_title_data
no_title1 = data["title"].str.contains(r"No Title$", regex = True)

#no_title_data
no_title_data = data[no_title1]

data["no_title"] = no_title1

filtered_data = data[data["no_title"]  == False ]

filtered_data["date"] = pd.to_datetime(filtered_data['pubdate'], errors = "coerce")

filtered_data["year"] = filtered_data["date"].dt.year
filtered_data["month"] = filtered_data["date"].dt.month_name()
filtered_data["day"] = filtered_data["date"].dt.day


C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  app.launch_new_instance()
C:\ProgramData\Anaconda3\lib\site-packages\ipykernel_launcher.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation:

In [7]:
filtered_data = filtered_data.rename(columns = {"Unnamed: 0": "index"})

In [8]:
filtered_data

,index,pubdate,title,abstract,link,author,news_desk,date,no_title,year,month,day
1,1,"Nov. 24, 1851",The Mission of Dr. Kinkel.,Editorial upon his mission,https://www.nytimes.com/1851/11/24/archives/th...,Editorial upon his mission,ARCHIVES,1851-11-24,False,1851,November,24
2,2,"Nov. 27, 1851",EUROPE.; ARRIVAL OF THE ASIA'S MAILS. History ...,NaN,https://www.nytimes.com/1851/11/27/archives/eu...,NaN,ARCHIVES,1851-11-27,False,1851,November,27
3,3,"Dec. 16, 1851",BANQUET OF THE PRESS TO LOUIS KOSSUTH. AT THE ...,Speech at the Press banquet,https://www.nytimes.com/1851/12/16/archives/ba...,Speech at the Press banquet,ARCHIVES,1851-12-16,False,1851,December,16
4,4,"Dec. 18, 1851",German Radicalism.,NaN,https://www.nytimes.com/1851/12/18/archives/ge...,NaN,ARCHIVES,1851-12-18,False,1851,December,18
5,5,"Dec. 29, 1851",KOSSUTH AT PHILADELPHIA.; Interview of the Com...,Letter to Phila dinner,https://www.nytimes.com/1851/12/29/archives/ko...,Letter to Phila dinner,ARCHIVES,1851-12-29,False,1851,December,29
...,...,...,...,...,...,...,...,...,...,...,...,...
177190,177190,"Dec. 30, 2019","Xinjiang, 7-Eleven, Hong Kong: Your Tuesday Br...",Here’s what you need to know.,https://www.nytimes.com/2019/12/30/briefing/xi...,By Melina Delkic,BRIEFING,2019-12-30,False,2019,December,30
177191,177191,"Dec. 30, 2019",Will Brexit Bring the Troubles Back to Norther...,As the United Kingdom confronts the prospect o...,https://www.nytimes.com/2019/12/30/magazine/br...,By James Angelos,MAGAZINE,2019-12-30,False,2019,December,30
177192,177192,"Dec. 30, 2019","Carlos Ghosn, Fallen Nissan Boss, Flees Japan ...","Mr. Ghosn, who was facing charges of financial...",https://www.nytimes.com/2019/12/30/business/ca...,"By Emily Flitter, Amy Chozick and Ben Dooley",BUSINESS,2019-12-30,False,2019,December,30
177193,177193,"Dec. 31, 2019",What I Learned in Avalanche School,I wanted to be prepared for the worst nature c...,https://www.nytimes.com/2019/12/31/magazine/av...,By Heidi Julavits,MAGAZINE,2019-12-31,False,2019,December,31


In [9]:
pdf1851_1980 = pd.read_csv("nyt_terrorism_pdftexts_proquest2.csv").sort_values(by=['index'])

In [10]:
pdf1851_1980

,index,pubdate,text
0,1,"Nov. 24, 1851",The Mission of Or. Linked. New Work Daily Tim...
20224,2,"Nov. 27, 1851","EUROPE.: <SPAN CLASS=""HIT"">ARRIVAL</SPAN> <SPA..."
1,2,"Nov. 27, 1851","EUROPE.: <SPAN CLASS=""HIT"">ARRIVAL</SPAN> <SPA..."
20225,3,"Dec. 16, 1851",of BANQUET of THE PRESS of LOUIS KOSSUTH. of T...
20226,4,"Dec. 18, 1851",German Radicalism. New Work Daily Times (18511...
...,...,...,...
20219,22850,"Dec. 28, 1980",Hoch Was Armed Guard In Old Jerusalem Talk Ne...
20220,22851,"Dec. 28, 1980",The Publishers' Dear Problems and a New Soluti...
20221,22852,"Dec. 28, 1980",Prodigious Daughter By CLAIRE STERLING New Wo...
20222,22853,"Dec. 29, 1980",ESSAY Ran's Christmas Coming By William Fire N...


In [19]:
import sys
import csv
maxInt = sys.maxsize

while True:
    # decrease the maxInt value by factor 10 
    # as long as the OverflowError occurs.

    try:
        csv.field_size_limit(maxInt)
        break
    except OverflowError:
        maxInt = int(maxInt/2)

In [21]:
online1981_2019 = pd.read_csv("nyt_online_texts.csv", encoding = 'utf-8', engine = "python", error_bad_lines=False)

Skipping line 4738: Expected 2 fields in line 4738, saw 4
Skipping line 5435: Expected 2 fields in line 5435, saw 4


In [29]:
online1981_2019.tail(100)

,index,texts
92056,99,"It has been the boast of the South, and especi..."
92057,98,The Address to the People of Virginia by the C...
92058,97,Gen. BANKS dealt a vigorous blow at the rebels...
92059,96,"Mr. GREGORY, the Irish advocate employed by th..."
92060,95,"Senator JOHNSON, of Tennessee, is reported on ..."
...,...,...
92151,4,The eagerness of those prints which addict the...
92152,3,In decorating the splendid saloon in which the...
92153,2,The steamer Asia reached Boston on Tuesday aft...
92154,1,While Mr. Robert Walker hands round his hat in...


In [49]:
filtered_data.merge(pdf1851_1980, on = "index", how = "outer")

,index,pubdate_x,title,abstract,link,author,news_desk,date,no_title,year,month,day,pubdate_y,text
0,1,"Nov. 24, 1851",The Mission of Dr. Kinkel.,Editorial upon his mission,https://www.nytimes.com/1851/11/24/archives/th...,Editorial upon his mission,ARCHIVES,1851-11-24,False,1851,November,24,"Nov. 24, 1851",The Mission of Or. Linked. New Work Daily Tim...
1,2,"Nov. 27, 1851",EUROPE.; ARRIVAL OF THE ASIA'S MAILS. History ...,NaN,https://www.nytimes.com/1851/11/27/archives/eu...,NaN,ARCHIVES,1851-11-27,False,1851,November,27,"Nov. 27, 1851","EUROPE.: <SPAN CLASS=""HIT"">ARRIVAL</SPAN> <SPA..."
2,2,"Nov. 27, 1851",EUROPE.; ARRIVAL OF THE ASIA'S MAILS. History ...,NaN,https://www.nytimes.com/1851/11/27/archives/eu...,NaN,ARCHIVES,1851-11-27,False,1851,November,27,"Nov. 27, 1851","EUROPE.: <SPAN CLASS=""HIT"">ARRIVAL</SPAN> <SPA..."
3,3,"Dec. 16, 1851",BANQUET OF THE PRESS TO LOUIS KOSSUTH. AT THE ...,Speech at the Press banquet,https://www.nytimes.com/1851/12/16/archives/ba...,Speech at the Press banquet,ARCHIVES,1851-12-16,False,1851,December,16,"Dec. 16, 1851",of BANQUET of THE PRESS of LOUIS KOSSUTH. of T...
4,4,"Dec. 18, 1851",German Radicalism.,NaN,https://www.nytimes.com/1851/12/18/archives/ge...,NaN,ARCHIVES,1851-12-18,False,1851,December,18,"Dec. 18, 1851",German Radicalism. New Work Daily Times (18511...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
176971,177190,"Dec. 30, 2019","Xinjiang, 7-Eleven, Hong Kong: Your Tuesday Br...",Here’s what you need to know.,https://www.nytimes.com/2019/12/30/briefing/xi...,By Melina Delkic,BRIEFING,2019-12-30,False,2019,December,30,NaN,NaN
176972,177191,"Dec. 30, 2019",Will Brexit Bring the Troubles Back to Norther...,As the United Kingdom confronts the prospect o...,https://www.nytimes.com/2019/12/30/magazine/br...,By James Angelos,MAGAZINE,2019-12-30,False,2019,December,30,NaN,NaN
176973,177192,"Dec. 30, 2019","Carlos Ghosn, Fallen Nissan Boss, Flees Japan ...","Mr. Ghosn, who was facing charges of financial...",https://www.nytimes.com/2019/12/30/business/ca...,"By Emily Flitter, Amy Chozick and Ben Dooley",BUSINESS,2019-12-30,False,2019,December,30,NaN,NaN
176974,177193,"Dec. 31, 2019",What I Learned in Avalanche School,I wanted to be prepared for the worst nature c...,https://www.nytimes.com/2019/12/31/magazine/av...,By Heidi Julavits,MAGAZINE,2019-12-31,False,2019,December,31,NaN,NaN


In [16]:
start = 2030
#end = filterdata.date[-1] < 1981-01-01
end = 2512 

In [17]:
'''
start = 22771
#end = filterdata.date[-1] < 1981-01-01
end = 22855
'''

'\nstart = 22771\n#end = filterdata.date[-1] < 1981-01-01\nend = 22855\n'

In [18]:
filtered_data = filtered_data[(filtered_data.index > start) & (filtered_data.index < end)].drop_duplicates()
#lost:4920 5326 5375 6981 9827 11723 13864 13934 14367 14368 14692:14698 14900 15135 15434 15636 15735 17209-10 17326 
#17469 18006 18225 18550 19175 19179 19365 19462 19812 19815 19816 20600 21241 21890 22212 881

In [31]:
filtered_data[filtered_data.year == 1981]

,index,pubdate,title,abstract,link,author,news_desk,date,no_title,year,month,day
22855,22855,"Jan. 1, 1981",AROUND THE NATION; Pastor Says He's in Hiding ...,UPI,https://www.nytimes.com/1981/01/01/us/around-t...,UPI,U.S.,1981-01-01,False,1981,January,1
22856,22856,"Jan. 1, 1981",ITALIAN ANTITERRORIST GENERAL IS SHOT AND KILL...,AP,https://www.nytimes.com/1981/01/01/world/itali...,AP,WORLD,1981-01-01,False,1981,January,1
22857,22857,"Jan. 1, 1981",2 SYRIAN JET FIGHTERS SHOT DOWN BY ISRAELIS OV...,Special to the New York Times,https://www.nytimes.com/1981/01/01/world/2-syr...,Special to the New York Times,WORLD,1981-01-01,False,1981,January,1
22858,22858,"Jan. 1, 1981",EXPLOSION AT NAIROBI HOTEL KILLS 5; POSSIBILIT...,By Pranay B. Gupte,https://www.nytimes.com/1981/01/01/world/explo...,By Pranay B. Gupte,WORLD,1981-01-01,False,1981,January,1
22859,22859,"Jan. 1, 1981",SALVADORAN VILLAGE GETS A TASTE OF WAR,"By Raymond Bonner, Special To the New York Times",https://www.nytimes.com/1981/01/01/world/salva...,"By Raymond Bonner, Special To the New York Times",WORLD,1981-01-01,False,1981,January,1
...,...,...,...,...,...,...,...,...,...,...,...,...
24828,24828,"Dec. 30, 1981",BRIEFING,By Phil Gailey and David Shribman,https://www.nytimes.com/1981/12/30/us/briefing...,By Phil Gailey and David Shribman,U.S.,1981-12-30,False,1981,December,30
24829,24829,"Dec. 30, 1981",F.B.I. DIRECTOR SAYS INCIDENTS OF TERRORISM AR...,UPI,https://www.nytimes.com/1981/12/30/us/fbi-dire...,UPI,U.S.,1981-12-30,False,1981,December,30
24830,24830,"Dec. 31, 1981",TURKS PRESENT SCHEDULE FOR RETURN TO DEMOCRACY,"By Marvine Howe, Special To the New York Times",https://www.nytimes.com/1981/12/31/world/turks...,"By Marvine Howe, Special To the New York Times",WORLD,1981-12-31,False,1981,December,31
24831,24831,"Dec. 31, 1981",GLIMPSES OF '81: MOMENTS IN A YEAR OF VIVID IM...,NaN,https://www.nytimes.com/1981/12/31/nyregion/gl...,NaN,NEW YORK,1981-12-31,False,1981,December,31


In [232]:
get_nyt_news_proquest_get_pdftexts("https://search-proquest-com.proxy.library.nyu.edu/hnpnewyorktimes/advanced", 
                                   filtered_data, filename)

In [ ]:
pdf_text = pd.read_csv(filename, error_bad_lines=False)

In [ ]:
pdf_text["date"] = pd.to_datetime(no_total12['pubdate'], errors = "coerce")


In [ ]:
pdf_text["date"].dropna()

In [ ]:
datadict = {"index" : index, "pubdate" : pubdate, "texts" : texts}

In [ ]:
total_text = pd.DataFrame(datadict)
total_text.head()

In [ ]:
total_text.to_csv("nyt_terrorism_pdftexts_test10.csv")


In [ ]:
words_clean_cloud_df(total_text, "texts", stem = False, outputdf = True)

In [ ]:
words_clean_cloud_df(no_total1, "texts", stem = False, outputdf = True)